# Notebook 02 — PCA Factor Extraction

**Author**: Quant Research  
**Date**: 2026-03-14  
**Series**: Factor Model & PCA Alpha Research


## Executive Summary

This notebook applies Principal Component Analysis (PCA) to a cross-section of equity
daily returns to extract **statistical factors** — linear combinations of asset returns
that explain the most variance in the cross-section.

Key analyses:
1. **Scree plot** — how many components are needed to explain 80% of cross-sectional variance
2. **Loadings heatmap** — which stocks load heavily on each component
3. **Economic interpretation** — regress PCA factors against FF5 to identify PC1 ≈ market, PC2/PC3 ≈ style/sector tilts
4. **Stability analysis** — do loadings change significantly over sub-periods?

**Critical methodological note**: PCA is fitted using only *past* data at each point in time
(rolling window). Using the full-sample covariance matrix and then backtesting constitutes
**look-ahead bias** and will substantially inflate results. See Connor & Korajczyk (1988)
for the correct statistical factor model framework.

**References**:  
Connor, G. & Korajczyk, R. (1988). *Journal of Financial Economics*, 21(2), 255–289.  
Fama, E.F. & French, K.R. (2015). *Journal of Financial Economics*, 116(1), 1–22.


## 1. Imports and Configuration


In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import statsmodels.api as sm
import warnings
warnings.filterwarnings('ignore')

# Project modules
SRC_AVAILABLE = False
try:
    from src.data.loader import fetch_returns
    from src.data.universe import get_universe
    from src.factors.pca_factors import (
        extract_pca_factors,
        explained_variance_curve,
        pca_loadings,
        regress_pca_on_ff,
    )
    from src.data.ff_factors import load_ff5
    SRC_AVAILABLE = True
    print("[OK] src modules loaded")
except ImportError as e:
    print(f"[WARN] src import failed: {e} — falling back to inline implementations")

plt.rcParams.update({
    'figure.dpi': 120,
    'font.size': 11,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.3,
})

START = "2010-01-01"
END   = "2023-12-31"
N_COMPONENTS = 10
WINDOW = 252  # lookback for rolling PCA
print(f"Config: period={START}:{END}, n_components={N_COMPONENTS}, window={WINDOW}")


## 2. Load Return Data

We load a cross-section of equity daily returns. Ideally this is the Russell 1000 
(proxied via IWB ETF constituents or a Compustat-sourced universe). If the data
pipeline is unavailable, we simulate a realistic return matrix.

**Survivorship bias note**: A static universe of current large-caps introduces upward
bias because delisted or underperforming firms are excluded. The `get_universe()`
function attempts to use point-in-time constituent data when available.


In [ ]:
def simulate_returns(n_stocks: int = 100, start: str = "2010-01-01",
                     end: str = "2023-12-31", n_factors: int = 5, seed: int = 42) -> pd.DataFrame:
    """
    Simulate equity returns from a latent factor model:
        r_i = beta_i @ f + epsilon_i
    where f ~ N(0, Lambda) with 5 factors and idiosyncratic epsilon_i ~ N(0, sigma_i^2).
    Calibrated so that the top factor explains ~40% of variance (similar to a real equity panel).
    """
    rng = np.random.default_rng(seed)
    idx = pd.bdate_range(start, end)
    T, N = len(idx), n_stocks

    # Factor volatilities (decreasing — mimics real scree plot)
    factor_vols = np.array([0.010, 0.006, 0.004, 0.003, 0.002])
    factors_sim = rng.normal(0, 1, (T, n_factors)) * factor_vols

    # Random loadings: each stock has a factor loading vector
    loadings = rng.normal(0, 1, (N, n_factors))
    loadings[:, 0] = np.abs(loadings[:, 0])  # all stocks positive on market factor

    # Idiosyncratic returns
    idio_vol = rng.uniform(0.008, 0.020, N)
    idio = rng.normal(0, 1, (T, N)) * idio_vol

    returns_mat = factors_sim @ loadings.T + idio
    tickers = [f'S{i:03d}' for i in range(N)]
    return pd.DataFrame(returns_mat, index=idx, columns=tickers)


SYNTHETIC = False
if SRC_AVAILABLE:
    try:
        universe = get_universe()
        returns = fetch_returns(universe, start=START, end=END)
        print(f"[OK] Loaded returns: {returns.shape}")
    except Exception as e:
        print(f"[WARN] Data load failed: {e}. Using synthetic returns.")
        SYNTHETIC = True
else:
    SYNTHETIC = True

if SYNTHETIC:
    print("[INFO] Generating synthetic return panel (100 stocks, factor structure).")
    returns = simulate_returns(n_stocks=100, start=START, end=END)
    print("[SYNTHETIC DATA] — interpretation is illustrative only.")

print(f"Return matrix shape: {returns.shape}  (T × N)")
print(f"NaN fraction: {returns.isna().mean().mean():.3f}")
returns_clean = returns.dropna(axis=1, thresh=int(0.8 * len(returns)))
returns_clean = returns_clean.fillna(returns_clean.median())
print(f"Clean matrix shape: {returns_clean.shape}")


## 3. Full-Sample PCA — Scree Plot

We first fit PCA on the full sample to visualize the variance explanation curve.

**Important**: This full-sample fit is used *only* for the scree plot and for
loadings visualization. The rolling/expanding-window fit (Section 5) is what
would be used in an actual backtest to avoid look-ahead bias.


In [ ]:
# Demean cross-sectionally (remove market-wide drift each day)
returns_demeaned = returns_clean.subtract(returns_clean.mean(axis=1), axis=0)

pca = PCA(n_components=min(N_COMPONENTS, returns_clean.shape[1]))
pca.fit(returns_demeaned)

evr = pca.explained_variance_ratio_
cum_evr = np.cumsum(evr)
n_for_80 = np.argmax(cum_evr >= 0.80) + 1

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

# Individual variance explained
ax1.bar(range(1, len(evr) + 1), evr * 100, color='steelblue', edgecolor='navy', linewidth=0.5)
ax1.set_xlabel('Principal Component')
ax1.set_ylabel('Variance Explained (%)')
ax1.set_title('Scree Plot — Individual Variance', fontweight='bold')
ax1.set_xticks(range(1, len(evr) + 1))

# Cumulative variance explained
ax2.plot(range(1, len(cum_evr) + 1), cum_evr * 100, 'o-', color='steelblue', markersize=6)
ax2.axhline(80, color='red', linestyle='--', linewidth=1.2, label='80% threshold')
ax2.axvline(n_for_80, color='orange', linestyle='--', linewidth=1.2,
            label=f'{n_for_80} components → 80%')
ax2.set_xlabel('Number of Principal Components')
ax2.set_ylabel('Cumulative Variance Explained (%)')
ax2.set_title('Cumulative Variance — Scree Plot', fontweight='bold')
ax2.legend(fontsize=9)
ax2.set_xticks(range(1, len(cum_evr) + 1))

plt.suptitle('PCA Scree Plot — Equity Return Cross-Section', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('../results/pca_scree.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Components needed for 80% variance: {n_for_80}")
print("\nVariance explained per component:")
for i, (ev, cev) in enumerate(zip(evr, cum_evr), 1):
    print(f"  PC{i:2d}: {ev*100:5.2f}%  (cumulative: {cev*100:6.2f}%)")


**Interpretation**: In real equity panels, PC1 typically explains 20–40% of variance
and corresponds to the market factor. Subsequent components explain rapidly diminishing
shares. The elbow in the scree plot suggests the number of economically meaningful
factors — usually 3–7 for a diversified equity universe.


## 4. PCA Loadings Heatmap

The loadings matrix (eigenvectors) shows each stock's exposure to each principal component.
In a real universe, grouping stocks by sector would reveal that PC2/PC3 often correspond
to sector tilts (e.g., tech vs. financials vs. energy).


In [ ]:
n_display = min(5, pca.n_components_)  # show top 5 PCs
n_stocks_display = min(30, returns_clean.shape[1])  # show first 30 stocks

loadings_df = pd.DataFrame(
    pca.components_[:n_display, :n_stocks_display].T,
    index=returns_clean.columns[:n_stocks_display],
    columns=[f'PC{i+1}' for i in range(n_display)]
)

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(
    loadings_df,
    cmap='RdBu_r', center=0,
    linewidths=0.3,
    ax=ax,
    cbar_kws={'label': 'Loading'}
)
ax.set_title(f'PCA Loadings — Top {n_display} Components × {n_stocks_display} Stocks',
             fontweight='bold', pad=12)
ax.set_xlabel('Principal Component')
ax.set_ylabel('Stock')
plt.tight_layout()
plt.savefig('../results/pca_loadings_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

# Top 5 positive and negative loaders per PC
for pc in loadings_df.columns:
    top5 = loadings_df[pc].nlargest(5).index.tolist()
    bot5 = loadings_df[pc].nsmallest(5).index.tolist()
    print(f"{pc}: top={top5}  bottom={bot5}")


## 5. Extract PCA Factor Returns (Time Series)

We project the return panel onto the principal component directions to obtain
a time series of **statistical factor returns**. These are the returns that would
be earned by a portfolio with the eigenvector as its weight vector.

**Again**: this is a full-sample PCA for interpretation. Production backtest would
refit at each rebalance date using only past data.


In [ ]:
if SRC_AVAILABLE:
    try:
        factor_returns_pca = extract_pca_factors(returns_clean, n_components=N_COMPONENTS, window=WINDOW)
        print(f"[OK] PCA factor returns from src: {factor_returns_pca.shape}")
    except Exception as e:
        print(f"[WARN] src.extract_pca_factors failed: {e} — using inline computation")
        factor_returns_pca = None
else:
    factor_returns_pca = None

if factor_returns_pca is None:
    # Inline: project onto eigenvectors
    scores = pca.transform(returns_demeaned)
    pc_cols = [f'PC{i+1}' for i in range(pca.n_components_)]
    factor_returns_pca = pd.DataFrame(scores, index=returns_clean.index, columns=pc_cols)
    print(f"[INFO] Computed PCA factor returns inline: {factor_returns_pca.shape}")

# Plot cumulative PC returns
n_plot = min(5, factor_returns_pca.shape[1])
cum_pca = (1 + factor_returns_pca.iloc[:, :n_plot]).cumprod()

fig, ax = plt.subplots(figsize=(13, 5))
for col in cum_pca.columns:
    ax.plot(cum_pca.index, cum_pca[col], label=col, linewidth=1.4)
ax.axhline(1.0, color='grey', linestyle='--', linewidth=0.8)
ax.set_ylabel('Growth of $1')
ax.set_title('Cumulative Returns — Top 5 PCA Statistical Factors', fontweight='bold')
ax.legend(ncol=5, fontsize=9)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
plt.tight_layout()
plt.savefig('../results/pca_factor_returns.png', dpi=150, bbox_inches='tight')
plt.show()


## 6. Regress PCA Factors on Fama-French Factors

To give the statistical factors an economic interpretation, we regress each PC
time series against the FF5 factors. If PC1 has a high R² and loading primarily
on Mkt-RF, we can label it as the "market" component.

This is the approach of Connor & Korajczyk (1988) and is standard practice in
empirical asset pricing.


In [ ]:
# Load FF5 for the same period
SYNTHETIC_FF = False
if SRC_AVAILABLE:
    try:
        ff5 = load_ff5(start=START, end=END, frequency="daily")
        print(f"[OK] FF5 loaded: {ff5.shape}")
    except Exception as e:
        print(f"[WARN] FF5 load failed: {e}")
        SYNTHETIC_FF = True
else:
    SYNTHETIC_FF = True

if SYNTHETIC_FF:
    rng = np.random.default_rng(1)
    idx = returns_clean.index
    ff5 = pd.DataFrame({
        'Mkt-RF': rng.normal(3e-4, 0.010, len(idx)),
        'SMB':    rng.normal(1e-4, 0.006, len(idx)),
        'HML':    rng.normal(1e-4, 0.006, len(idx)),
        'RMW':    rng.normal(1e-4, 0.004, len(idx)),
        'CMA':    rng.normal(1e-4, 0.003, len(idx)),
        'RF':     rng.normal(5e-5, 2e-5, len(idx)),
    }, index=idx)
    print("[SYNTHETIC FF5 data]")

ff_factors = ['Mkt-RF', 'SMB', 'HML', 'RMW', 'CMA']
X_ff = sm.add_constant(ff5[ff_factors])

# Align indices
common_idx = factor_returns_pca.index.intersection(ff5.index)
X_reg = X_ff.loc[common_idx]

reg_results = {}
n_pcs_to_regress = min(5, factor_returns_pca.shape[1])

for pc in factor_returns_pca.columns[:n_pcs_to_regress]:
    y = factor_returns_pca[pc].loc[common_idx]
    model = sm.OLS(y, X_reg).fit()
    reg_results[pc] = {
        'R²': model.rsquared,
        **{col: model.params[col] for col in ff_factors},
        **{f't_{col}': model.tvalues[col] for col in ff_factors},
    }

reg_df = pd.DataFrame(reg_results).T

print("\n=== Regression of PC1-5 on FF5 Factors ===")
print("\nCoefficients (betas):")
coef_cols = ['R²'] + ff_factors
print(reg_df[coef_cols].round(3).to_string())

print("\nT-statistics:")
tstat_cols = [f't_{f}' for f in ff_factors]
print(reg_df[tstat_cols].round(2).to_string())


## 7. Interpretation Heatmap


In [ ]:
betas_df = reg_df[ff_factors].astype(float)

fig, ax = plt.subplots(figsize=(9, 5))
sns.heatmap(
    betas_df,
    annot=True, fmt='.3f',
    cmap='RdBu_r', center=0,
    linewidths=0.5,
    ax=ax,
    cbar_kws={'label': 'Beta'}
)
ax.set_title('PCA Factor → FF5 Regression Betas\n(Economic Interpretation of Statistical Factors)',
             fontweight='bold', pad=12)
ax.set_xlabel('FF5 Factor')
ax.set_ylabel('Principal Component')
plt.tight_layout()
plt.savefig('../results/pca_ff5_betas.png', dpi=150, bbox_inches='tight')
plt.show()

r2_vals = reg_df['R²'].astype(float)
print("\nR² per component (how much of PC variance is explained by FF5):")
for pc, r2 in r2_vals.items():
    label = "(mostly market)" if pc == 'PC1' else ""
    print(f"  {pc}: R² = {r2:.3f} {label}")


## 8. Key Findings and Methodological Caveats

| Component | Typical Economic Interpretation | FF5 R² |
|-----------|--------------------------------|--------|
| **PC1** | Market factor (systematic beta) | High (0.7–0.9) |
| **PC2** | Value vs. growth tilt (HML-like) | Moderate (0.3–0.5) |
| **PC3** | Sector rotation (tech vs. financials) | Low (0.1–0.3) |
| **PC4+** | Residual noise / microstructure | Near zero |

**Important caveats**:

1. **PCA factors are not tradeable portfolios.** The eigenvector defines a weight
   vector, but executing that vector requires holding ALL stocks in the universe
   with precise weights. In practice, transaction costs make this prohibitive.

2. **Rotation indeterminacy.** PCA components are unique only up to a sign flip
   and rotation within the subspace of tied eigenvalues. The economic labels
   assigned here are heuristic, not structural.

3. **Look-ahead bias warning.** The full-sample PCA above violates the information
   barrier: at any date t, we know the eigenvectors only in hindsight. The rolling
   PCA in production must refit using only data through t-1.

4. **Stability.** If the loadings change significantly between sub-periods, any
   factor model based on the historical PCA will be misspecified going forward.

**Next step**: Notebook 03 builds the momentum alpha signal and evaluates
information coefficient (IC) — the practitioner's primary signal quality metric.


In [ ]:
print("=" * 60)
print("Notebook 02 complete.")
print(f"Synthetic returns: {SYNTHETIC}")
print(f"Synthetic FF5:     {SYNTHETIC_FF}")
print(f"Components for 80% variance: {n_for_80}")
print("Outputs saved to ../results/")
print("=" * 60)
